<a href="https://colab.research.google.com/github/sampabiet90/AIML/blob/LogicMojo-AI-ML-April26-sampa90/image_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
from datasets import load_dataset
from copy import deepcopy
from pathlib import Path
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

In [57]:
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("Device:", DEVICE)

Device: cpu


In [58]:
ds = load_dataset("AlvaroVasquezAI/Animal_Image_Classification_Dataset")

In [59]:
ds
print(ds["train"].features)


{'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['cats', 'dogs', 'snakes'])}


In [60]:
SEED = 42
split = ds["train"].train_test_split(
    test_size=0.2,
    seed=SEED
)

train_ds = split["train"]
temp_ds = split["test"]

# Split temporary 20% into validation and test
split_test = temp_ds.train_test_split(
    test_size=0.5,
    seed=SEED
)

valid_ds = split_test["train"]
test_ds = split_test["test"]

print("Training images:", len(train_ds))
print("Validation images:", len(valid_ds))
print("Testing images:", len(test_ds))

Training images: 2400
Validation images: 300
Testing images: 300


In [61]:
# ============================================================
# 3. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])





#

In [62]:
class AnimalDataset(Dataset):

    def __init__(self, dataset, transform=None):

        self.dataset = dataset
        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, index):

        example = self.dataset[index]

        # Get image directly from Hugging Face dataset
        image = example["image"]

        # Get label directly from Hugging Face dataset
        label = example["label"]

        # Convert to RGB
        image = image.convert("RGB")

        # Apply transform only when image is requested
        if self.transform is not None:
            image = self.transform(image)

        return image, torch.tensor(
            label,
            dtype=torch.long
        )


# CREATE PYTORCH DATASETS

train_dataset = AnimalDataset(
    train_ds,
    transform=train_transform
)

test_dataset = AnimalDataset(
    test_ds,
    transform=test_transform
)

valid_dataset = AnimalDataset(
    valid_ds,
    transform=test_transform
)


# ============================================================
# 6. DETERMINISTIC DATALOADER
# ============================================================

def make_loader(dataset, shuffle=False, seed=SEED):

    generator = torch.Generator().manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=16,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0
    )

# ============================================================
# 7. CREATE LOADERS
# ============================================================

train_loader = make_loader(
    train_dataset,
    shuffle=True,
    seed=SEED
)

test_loader = make_loader(
    test_dataset,
    shuffle=False,
    seed=SEED
)

valid_loader = make_loader(
    valid_dataset,
    shuffle=False,
    seed=SEED
)



In [63]:
# WHAT: Define one training epoch and one deterministic evaluation pass.
# WHY: All strategies should use the same loss and metric calculations.
# OUTPUT: Reusable functions returning loss, accuracy, labels, and predictions.

def train_one_epoch(model, loader, optimizer, *, use_augmentation):
    model.train()

    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, count = 0.0, 0, 0
    for raw_images, labels in loader:

        inputs = raw_images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        count += labels.size(0)
    return loss_sum / count, correct / count

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, count = 0.0, 0
    labels_all, predictions_all = [], []
    for raw_images, labels in loader:
        inputs = raw_images.to(DEVICE)
        labels_device = labels.to(DEVICE)
        logits = model(inputs)
        loss = criterion(logits, labels_device)

        loss_sum += loss.item() * labels.size(0)
        count += labels.size(0)
        labels_all.append(labels)
        predictions_all.append(logits.argmax(1).cpu())

    labels_array = torch.cat(labels_all).numpy()
    predictions_array = torch.cat(predictions_all).numpy()
    return {
        "loss": loss_sum / count,
        "accuracy": float((labels_array == predictions_array).mean()),
        "labels": labels_array,
        "predictions": predictions_array,
    }

In [64]:
# WHAT: Train a model, keep its best validation state, and stop when improvement stalls.
# WHY: The last epoch is not automatically the best model.
# OUTPUT: Restored best weights, learning history, and summary metrics.

def fit(
    model,
    train_loader,
    valid_loader,
    optimizer,
    *,
    epochs,
    use_augmentation,
    patience=6,
):
    history = {"train_loss": [], "valid_loss": [], "valid_accuracy": []}
    best_state = deepcopy(model.state_dict())
    best_loss = float("inf")
    best_epoch = 0
    stale = 0
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, _ = train_one_epoch(
            model,
            train_loader,
            optimizer,
            use_augmentation=use_augmentation
        )
        valid = evaluate(model, valid_loader)
        history["train_loss"].append(train_loss)
        history["valid_loss"].append(valid["loss"])
        history["valid_accuracy"].append(valid["accuracy"])

        if valid["loss"] < best_loss - 1e-4:
            best_loss = valid["loss"]
            best_state = deepcopy(model.state_dict())
            best_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    best_valid = evaluate(model, valid_loader)
    return {
        "history": history,
        "best_epoch": best_epoch,
        "valid_loss": best_valid["loss"],
        "valid_accuracy": best_valid["accuracy"],
        "seconds": time.perf_counter() - start,
    }

In [65]:
from torchvision.models import resnet18, ResNet18_Weights
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms


weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
preprocess = weights.transforms()

# --------------------------------------------------
# 3. Replace final layer for 3 animal classes
# --------------------------------------------------
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 3)

#freeze the backbone

for param in model.parameters():
    param.requires_grad = False

# Keep only the new classification head trainable
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(DEVICE)


# ============================================================
# 17. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# 18. OPTIMIZER FOR CLASSIFIER
# ============================================================

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=0.001
)


# ============================================================
# 19. TRAIN ONLY THE NEW CLASSIFIER
# ============================================================

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

outputs = model(images)

print("Output:", outputs.shape)

frozen_result = fit(
    model,
    train_loader,
    valid_loader,
    optimizer=optimizer,
    epochs=1,
    use_augmentation=False,
    patience=6,
)



print(
    f"\nFrozen model: "
    f"best epoch={frozen_result['best_epoch']}, "
    f"validation accuracy="
    f"{frozen_result['valid_accuracy']:.3f}"
)


final_frozen = evaluate(
    model,
    test_loader
)

print(
    f"\nFrozen model: "
    f"best epoch={frozen_result['best_epoch']}, "
    f"validation accuracy="
    f"{frozen_result['valid_accuracy']:.3f}, "
    f"test accuracy="
    f"{final_frozen['accuracy']:.3f}"
)





Images: torch.Size([16, 3, 160, 160])
Labels: torch.Size([16])
Output: torch.Size([16, 3])

Frozen model: best epoch=1, validation accuracy=0.957

Frozen model: best epoch=1, validation accuracy=0.957, test accuracy=0.947
